# BUSI2066 · Week 6 — From a Sample to a Population


### Before you start

Drag in **this notebook**, **`marketing_campaign.csv`** and **`loans_income.csv`**, and select the kernel.

### Today

Everything so far described **the file in front of you**. Today you start making claims about the
people who are **not** in it — which is the only reason anybody runs a survey.

Three ideas, and the third is the one you will actually use in your project:

1. **A sample is not the population**, and different samples give different answers
2. **Sample means behave far better than the data does** — this is why any of it works
3. **Bootstrapping** — how to get a confidence interval from the one sample you actually have

### How today runs

We go through **all the slides first**. Then there is **one block in class** for this notebook:

1. Run **Parts 1 to 4** top to bottom, reading as you go
2. Do **questions 4 and 6** of the exercise

Questions 1–3, 5 and 7 and the ⭐ sections are homework.

### The rule

> **AI explains. It does not write** — except where marked ⭐.

**One loop appears today**, in Part 2. You are not expected to write it. You are expected to read
it, change one number in it, and say what happened.

---
# Part 0 · Set up

In [ ]:
%pip install -q pandas seaborn scipy statsmodels

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

df = pd.read_csv('marketing_campaign.csv')
loans = pd.read_csv('loans_income.csv').rename(columns={'x': 'income'})

print(df.shape, loans.shape)

`numpy` is new — imported as `np`, by the same convention as `pd` and `sns`. It is the package that
does the random drawing.

Two files today, doing **two different jobs**:

| File | Its job today |
|---|---|
| `loans_income.csv` — 50,000 real incomes | **A pretend population.** We use it to show you something you can never normally see |
| `marketing_campaign.csv` — 1,344 customers | **A real sample.** It is what you would actually have |

---
# Part 1 · A sample is not the population

Pretend those 50,000 loan applicants are the entire population, and that — as in real life — you
cannot see it. You can only afford to ask 100 of them.

In [ ]:
print(loans['income'].mean())      # the TRUE population mean - normally invisible to you

**68,760.52.** Remember that number. It is the thing every method today is trying to estimate.

Now take a sample of 100 and look at its mean.

In [ ]:
rng = np.random.RandomState(1)

sample = rng.choice(loans['income'], 100)
print(sample.mean())

`RandomState(1)` is a **seed**. Random numbers on a computer are not really random — they come from
a formula, and the seed is where the formula starts. Give it the same seed and you get the same
"random" numbers, which is what makes your results reproducible and your marker's life possible.

Run the cell below. It is the same line three times — only the seed changes: 1, 2, 3.

In [ ]:
print(1, np.random.RandomState(1).choice(loans['income'], 100).mean())
print(2, np.random.RandomState(2).choice(loans['income'], 100).mean())
print(3, np.random.RandomState(3).choice(loans['income'], 100).mean())

Three samples from the same population, three different answers, **none of them exactly 68,760.52**.

That gap between a sample mean and the true mean is not a mistake and it is not sloppiness. It is
**sampling variability**, and it is unavoidable. The whole of statistical inference is the business
of putting a size on it.

⚠️ **This is your project's problem, exactly.** You will collect 50 to 80 responses and report an
average. The average of *the next* 50 people would be different. Your job is not to pretend
otherwise — it is to say **how much** different it might be.

---
# Part 2 · Sample means behave better than data does

Here is the idea that makes everything else work.

Same 50,000 incomes, still pretending they are the whole population. We run **three experiments**.
Each one makes **1,000 numbers**, and each becomes one panel of the picture in §2.1.

| Panel | Each time | How often | We get |
|---|---|---|---|
| 1 · blue | Pick **1 person** at random. Write down their income | 1,000 times | **1,000 incomes** |
| 2 · green | Pick **5 people** at random. Work out their **mean** income | 1,000 times | **1,000 means** |
| 3 · red | Pick **20 people** at random. Work out their **mean** income | 1,000 times | **1,000 means** |

Panel 1 shows how much **people** differ. Panels 2 and 3 show how much **means** differ.

Doing something 1,000 times needs a loop.

📖 **Read-level code.** Look at it, understand what it does, change a number. Do not memorise it.

In [ ]:
rng = np.random.RandomState(22)

means_of_5 = [rng.choice(loans['income'], 5).mean() for _ in range(1000)]

print(len(means_of_5))
print(means_of_5[:3])

Read it right to left:

- `for _ in range(1000)` — **do this a thousand times.** The `_` is a throwaway name, used when you
  do not care which repetition you are on
- `rng.choice(loans['income'], 5)` — draw **5** incomes at random
- `.mean()` — take their average
- the square brackets `[ ]` collect the thousand answers into a list

So `means_of_5` holds **1,000 sample means, each from a sample of 5 people** — panel 2.

📖 `means_of_5[:3]` in the cell above shows **the first 3 items** of the list — the same idea as `head(3)`, for a list.

Now the same thing with samples of 20 — panel 3.

In [ ]:
rng = np.random.RandomState(33)

means_of_20 = [rng.choice(loans['income'], 20).mean() for _ in range(1000)]

print(len(means_of_20))

## 2.1 Look at all three together

📖 **Read-level code.** The cell below draws three charts in one picture. New pieces, one line each:

- `plt.subplots(3, 1, ...)` — make **3 charts stacked in 1 column**. `axes` holds the three of them
- `axes[0]`, `axes[1]`, `axes[2]` — the top, middle and bottom chart. Counting starts at 0
- `ax=axes[0]` — tells `histplot` **which** chart to draw on
- `sharex=True` — all three use **the same x-axis**, so you can compare them by eye
- `plt.tight_layout()` — moves the titles so they do not overlap

**Panel 1** is **1,000 single incomes** drawn from the 50,000 — a picture of the data itself.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)

raw = np.random.RandomState(11).choice(loans['income'], 1000)
sns.histplot(raw, bins=40, ax=axes[0], color='steelblue')
axes[0].set_title('Panel 1 · 1,000 single incomes')

sns.histplot(means_of_5, bins=40, ax=axes[1], color='seagreen')
axes[1].set_title('Panel 2 · 1,000 means of 5 people')

sns.histplot(means_of_20, bins=40, ax=axes[2], color='indianred')
axes[2].set_title('Panel 3 · 1,000 means of 20 people')

axes[2].set_xlabel('Income (USD)')
plt.tight_layout()
plt.show()

**Three panels, same x-axis. Going down, one thing stays and two things change.**

1. **Same centre.** All three sit around the true mean, 68,761.
2. **Narrower.** Means of 20 cluster far more tightly around the truth than means of 5. A mean of 20
   people is rarely far from the truth.
3. **More bell-shaped.** Panel 1 is the skewed income distribution you met last week. Panel 3 is close
   to a bell curve — even though the data it came from is not.

Put numbers on it.

📖 `np.std(means_of_5, ddof=1)` is the standard deviation of a **list**. `ddof=1` makes it divide by
**n − 1**, the sample rule from week 5 — the same thing pandas' `.std()` does for a column.

In [ ]:
print("data       — sd:", round(loans['income'].std(), 1),
      " skew:", round(stats.describe(loans['income']).skewness, 3))
print("means of 5 — sd:", round(np.std(means_of_5, ddof=1), 1),
      " skew:", round(stats.describe(means_of_5).skewness, 3))
print("means of 20— sd:", round(np.std(means_of_20, ddof=1), 1),
      " skew:", round(stats.describe(means_of_20).skewness, 3))

| | Standard deviation | Skewness |
|---|---|---|
| The data — all 50,000 incomes | 32,872 | **1.049** — clearly skewed |
| Means of 5 | 14,760 | 0.527 |
| Means of 20 | **7,270** | **0.332** — much closer to symmetric |

The first row uses **all 50,000** incomes. The 1,000 incomes in the top panel give almost the same
answer — sd 32,843, skew 1.15.

That is the **central limit theorem**, happening in front of you: *whatever shape the data has, the
distribution of its sample means tends toward normal as the sample gets bigger — and tighter.* The rule
of thumb is a sample of **more than 30**. It works for other statistics too, like a proportion.

**It is the reason a skewed variable does not stop you doing statistics.** You are almost never
testing a data value. You are testing a mean, and means are well behaved.

## 2.2 Four words, and the standard error

| Word | What it means | Where it is in the picture |
|---|---|---|
| **Statistic** | One number that sums up a sample. The **mean** is one | Each of the 1,000 means in panels 2 and 3 |
| **Data distribution** | How the **values** in one sample spread out | **Panel 1** — 1,000 incomes. A histogram of your survey column looks like this |
| **Sampling distribution** | How a **statistic** spreads out over many samples | **Panels 2 and 3** — 1,000 means. In real life you never see this: you have one sample |
| **Standard error (SE)** | The **standard deviation of the sampling distribution** | **Panel 3: 7,270.** A mean of 20 people usually misses the true mean by about 7,000 |

⚠️ **Standard deviation** = how different people are from each other (32,872). **Standard error** =
how far your mean may be from the truth (7,270 with 20 people). Do not report one where the other belongs.

You do not need 1,000 samples to get the standard error. There is a formula:

> **SE = σ / √n**

- **σ** ("sigma") — the population standard deviation. Here, 32,872
- **n** — the sample size

In real life you never know σ, so you use **s**, your sample's standard deviation: **SE ≈ s / √n**.

Below, `np.sqrt` is the square root. Three plain lines — the same formula for samples of 5, 20 and 80.

In [ ]:
sigma = loans['income'].std()      # σ, the population standard deviation

print(sigma / np.sqrt(5))
print(sigma / np.sqrt(20))
print(sigma / np.sqrt(80))

**4 × the people → ½ the standard error.** The true mean is **μ = 68,761**:

| n | SE = σ / √n | Most sample means land in | Off by |
|---|---|---|---|
| 5 | 14,701 | 54,060 → 83,461 | ± 21% |
| 20 | 7,350 (÷ 2) | 61,410 → 76,111 | ± 11% |
| 80 | 3,675 (÷ 2) | **65,085 → 72,436** | **± 5%** |

"Most" = about 2 samples in 3, the ones within 1 SE. **With 80 people, your mean is usually within 5%
of the truth.**

Look at the **√n** on the bottom of the formula. To halve your standard error you need **four times**
the data, not twice. That is why the project guidelines ask for 80 responses rather than 50, and also
why they do not ask for 500.

## 2.3 The law of large numbers

One more thing the 1,000 means can tell you. Where is their **centre**?

In [ ]:
print(np.mean(means_of_20))          # the average of the 1,000 sample means
print(loans['income'].mean())        # the true population mean

📖 `np.mean(...)` is the average of a list — like `.mean()` on a column.

**68,817** against the true **68,760.52**. That is the **law of large numbers**: the mean of many sample means
lands on the true mean. Any one sample can miss; the average of many does not.

📌 **We could only do any of this because we had the population.** You never will. Part 3 is what you
do instead.

---
# Part 3 · Bootstrapping — the trick that saves you

Back to reality. You have **one** sample, and no population.

`marketing_campaign.csv` is 1,344 customers *"randomly drawn from a large population"* — that is how
the data was described when it was set. So treat it the way you will treat your own survey: one
sample, one shot.

The 1,328 customers with an income are **a sample, not the population**. The population is the
retailer's **whole customer base**, which we do not have.

In [ ]:
income = df['Income'].dropna()

print(len(income))
print(income.mean())

**1,328 customers, mean income 51,466.20.**

That single number is a **point estimate** — our best guess at the population mean. On its own it is
almost useless, because it comes with no sense of how wrong it might be.

## 3.1 The idea

We cannot draw more samples from the population. But we can **draw new samples from our sample**:

- **The same size** as the sample: from 1,328 customers, draw **1,328** again. (From your 65 responses, draw 65)
- **With replacement**: after each pick, the person goes back in and can be picked again. So one
  customer can appear twice and another not at all

Here is one resample of **six made-up customers**, A to F — the same as resample 1 on slide 19.

In [ ]:
customers = ['A', 'B', 'C', 'D', 'E', 'F']

print(np.random.RandomState(3).choice(customers, 6))     # one resample: 6 picks, with replacement

**C A B D A A** — still 6 picks, but A three times and E and F not at all. Change the seed and you get
a different resample.

**Why the same size?** The standard error depends on n. Draw fewer and you measure the SE of a smaller sample.

Take each resample's mean. Do it **10,000 times**. The spread of those 10,000 means estimates the spread
of real sample means. `bootstrap()` does all of it in one call.

It sounds like cheating. It is not, and it works.

In [ ]:
from scipy.stats import bootstrap

result = bootstrap(data=[income],
                   statistic=np.mean,
                   n_resamples=10000,
                   confidence_level=0.95,
                   rng=1)

print("standard error:", result.standard_error)
print("95% CI:", result.confidence_interval)

| | |
|---|---|
| **Standard error** | **597.85** — the standard deviation of the 10,000 resample means |
| **95% confidence interval** | **50,287 → 52,632** |

Check it against the formula from §2.2:

In [ ]:
print(income.std() / np.sqrt(len(income)))

**596.12** from the formula against **597.85** from 10,000 resamples. Two utterly different methods,
the same answer to within two dollars. **That agreement is the point** — the bootstrap is not a
trick, it is arriving at the truth the long way round, and it keeps working in cases where no
formula exists.

## 3.2 What a confidence interval actually says

> **"We are 95% confident the true average income in the population is between 50,287 and 52,632."**

That is the sentence. Two warnings about it:

- It is **not** "95% of customers earn between those numbers." It is about **the average**, not
  about people. The range for people is far wider — `income.describe()` shows incomes from 2,447 to 160,803.
- A wider interval is not a worse result. It is an **honest** result from less data.

## 3.3 Is the bootstrap distribution normal?

§2 claimed sample means tend toward normal. The 10,000 resample means are sitting right there, so
check the claim instead of believing it.

In [ ]:
resample_means = result.bootstrap_distribution

plt.figure(figsize=(7, 4))
sns.histplot(resample_means, bins=50, color='mediumpurple')
plt.xlabel('Resample mean income')
plt.title('10,000 bootstrap resample means')
plt.show()

print("sd of the 10,000 means:", np.std(resample_means, ddof=1))     # = the standard error above
print("skewness:", stats.describe(resample_means).skewness)
print("kurtosis:", stats.describe(resample_means).kurtosis)

In [ ]:
plt.figure(figsize=(6, 5))
stats.probplot(resample_means, plot=plt)
plt.title('QQ plot — bootstrap resample means')
plt.show()

**sd 597.85** — exactly the standard error `bootstrap()` printed. That is all a bootstrap standard error is.

**Skewness 0.029. Kurtosis 0.038.** Both effectively zero, and the QQ plot is a straight line.

Now the income data itself, for comparison:

In [ ]:
print("skewness of the income data itself:", stats.describe(income).skewness)

**0.28** — the data is a little skewed. **The means of it are normal anyway.** That is the central limit theorem again, now on your own data rather than on a pretend
population — and it is what lets you use a t-test in Part 4.

## 3.4 Where the interval comes from — trim the ends

`bootstrap()` gave you a 95% interval. Here is the idea behind it. Line up the 10,000 resample means.
For a **95%** interval, cut **2.5%** off each end. For a **90%** interval, cut **5%** off each end.
The two cut points are the interval.

📖 `np.percentile(resample_means, [2.5, 97.5])` finds the two cut points: 2.5% of the values sit below
the first, 97.5% sit below the second.

In [ ]:
print("95% CI:", np.percentile(resample_means, [2.5, 97.5]))
print("90% CI:", np.percentile(resample_means, [5, 95]))

| | Cut off each end | Interval |
|---|---|---|
| **95%** | 2.5% | **50,296 → 52,636** |
| **90%** | 5% | **50,484 → 52,466** |

**90% is narrower than 95%.** Less confidence, tighter range. And the 95% answer is a few dollars from
`bootstrap()`'s 50,287 → 52,632 — scipy adds a small correction to the cut points. Same idea.

## 3.5 The interval by hand — x̄ ± 2 SE

There is also a formula, and you should be able to use it:

> **95% CI = x̄ − 2 · s/√n  to  x̄ + 2 · s/√n**

- **x̄** ("x-bar") — the **sample mean**, the mean of the data you collected. You always know it
- **s** — the sample standard deviation · **n** — the sample size · **s/√n** — the standard error

**Why 2?** Think of panel 3 in §2.1: about **95% of sample means land within 2 SE of the true mean**.
So if you go 2 SE either side of **your** mean, you catch the true mean about 95% of the time. (The exact
number is 1.96 for a big sample, and a bit more for a small one.)

In [ ]:
se = income.std() / np.sqrt(len(income))

print("SE:", se)
print("lower:", income.mean() - 2 * se)
print("upper:", income.mean() + 2 * se)

**50,274 → 52,658.** Close to the bootstrap's 50,287 → 52,632. Two routes, almost the same answer.

Now the same thing from **only 50 customers** — your project's sample size.

In [ ]:
small = np.random.RandomState(5).choice(income, 50)
se_small = np.std(small, ddof=1) / np.sqrt(50)

print("mean of 50:", small.mean())
print("lower:", small.mean() - 2 * se_small)
print("upper:", small.mean() + 2 * se_small)

**42,768 → 54,342** — about **five times as wide.** A smaller sample means more uncertainty, so a wider
interval. Nothing went wrong; that is simply what 50 people can tell you.

---
# Part 4 · Your first statistical test

A confidence interval estimates a value. A **hypothesis test** answers a yes-or-no question about it.

Here is one worth asking. Suppose the industry benchmark for this kind of customer base is an
average income of **50,000**. Is this retailer's customer base different from the benchmark?

## 4.1 Write the hypotheses first — always

| | |
|---|---|
| **H₀** | The population mean income **is** 50,000 |
| **H₁** | The population mean income **is not** 50,000 |

H₁ names no direction, so this is a **two-sided** test. Week 4 said you never prove H₁ — you reject
H₀ or you fail to. §4.3 does exactly that.

## 4.2 Work out t by hand first

The t-score is **how many standard errors the sample mean is from the H₀ value**:

> **t = (x̄ − μ) / SE**

### The p-value

> **p-value: the probability, assuming H₀ is true, of getting a test statistic at least as extreme as the
> one observed.** Here the test statistic is t.

**In plain words, for this test:** if H₀ were true, how likely is a sample mean at least as far from the H₀
value as ours? Ours is 51,466 − 50,000 = 1,466 away. In standard errors that is t = 1,466 ÷ 596 = 2.46. So
p = the chance of t ≥ 2.46 or t ≤ −2.46 — a mean of 51,466 or above, or 48,534 or below. **Small p** → our
sample would be very unusual if H₀ were true → reject H₀.

### The 2 SE rule

If H₀ is true, a sample mean more than **2 SE** away happens less than 5% of the time. That 5% cut-off is
the **significance level**. In t-scores: **|t| > 2**, which means t above 2 or below −2.

⚠️ **|t| > 2 → p < 0.05 only works for about 60 people or more.** Smaller samples need a bigger t: 2.05 for
30, 2.26 for 10, 3.18 for 4. Why: with a small sample, s is only a rough guess of σ, so the SE is a rough
guess too, and the t-curve has fatter tails to allow for that. For your project's 65–80 responses the rule
works. Close to 2? Read p from `ttest_1samp`.

`se` is the standard error from §3.5.

In [ ]:
print((income.mean() - 50000) / se)

**2.46** — more than 2, so we expect p < 0.05. Now let scipy do it properly.

## 4.3 Run it

In [ ]:
from scipy.stats import ttest_1samp

result = ttest_1samp(a=income,
                     popmean=50000,
                     alternative='two-sided')

print("t statistic:", result.statistic)
print("p-value:", result.pvalue)
print("95% CI:", result.confidence_interval(confidence_level=0.95))

Three lines of input, three numbers out:

| Argument | What it is |
|---|---|
| `a=income` | your data |
| `popmean=50000` | **the value in H₀** — the thing you are testing against |
| `alternative='two-sided'` | H₁ has no direction. The others are `'less'` and `'greater'` |

## 4.4 Reading the output

| | Value | What it means |
|---|---|---|
| **t** | **2.46** | The sample mean is 2.46 standard errors above 50,000 — the same as your hand calculation |
| **p** | **0.0140** | **If H₀ were true**, there is a 1.4% chance of a sample this far off |
| **95% CI** | **50,297 → 52,636** | Where the true mean probably sits |

**The rule: if p < 0.05, reject H₀.**

Write the conclusion in this form:

> **Since p = 0.0140 < 0.05, we reject H₀. At the 5% significance level, there is sufficient evidence to
> conclude that the population mean income is different from 50,000.**

If p had been 0.05 or more: *"We fail to reject H₀. There is not sufficient evidence to conclude that the
population mean income is different from 50,000."* Never write "accept H₀".

Now notice the confidence interval does the same job from a different angle: **50,000 is not inside
it.** That is not a coincidence — a two-sided test at 5% and a 95% confidence interval are the same
statement written two ways. And compare it to the bootstrap interval in §3.1: 50,287 → 52,632 from
resampling, 50,297 → 52,636 from the t-test, 50,274 → 52,658 by hand. **Three methods, one answer.**

## 4.5 What p is not

This is the most misread number in statistics, so be precise about it.

| ❌ p is not | ✅ p is |
|---|---|
| the probability that H₀ is true | the probability of data this extreme, **assuming H₀ is true** |
| the probability you are wrong | a statement about the data, not about you |
| a measure of how big the effect is | entirely separate from size — see below |

⚠️ **p < 0.05 does not mean the difference matters.** Here the difference is about 1,466 on an
average income of 51,466 — under 3%. It is *statistically detectable*, which is a claim about
evidence, not about importance. With a big enough sample, almost anything becomes significant.

**Your report needs both: the p-value, and whether the size of the difference means anything to the
brand.**

---
# 🛠 Exercise · Estimate and test

Seven questions. Expected answers are shown where there is one.

**In class: questions 4 and 6.** Questions 1–3, 5 and 7 are homework.

**Questions 4 to 7 use `Age`.** This file has no `Age` column — you built it in week 4. Build it again
the same way, with the same `THIS_YEAR = 2026`, so your answers match the ones shown. Run this cell:

In [ ]:
THIS_YEAR = 2026

df['Age'] = THIS_YEAR - df['Year_Birth']

print(df['Age'].mean())

**55.74.** The 133-year-old from week 4 is still in the file. Leave them in for this exercise.

**1 · How different is a small sample?** Draw three samples of **30** from `loans['income']`, using
seeds 7, 8 and 9, and print each mean.

Then say how far the worst one is from the true population mean of 68,760.

_(your answer)_

In [ ]:
# Your code here

**2 · Change one number in the loop.** Copy the `means_of_20` cell from §2, change the sample size
from 20 to **100**, and keep seed 33.

What are the standard deviation and skewness of the result? Compare to means of 20.

```
sd  ≈ 3191
skew ≈ 0.098
```

In [ ]:
# Your code here

**3 · Check it against the formula.** For a sample of 100, what does `sd / sqrt(n)` predict the
standard error should be? Does your answer to question 2 agree?

_(your answer)_

In [ ]:
# Your code here

**4 · Bootstrap a different column.** Build a 95% confidence interval for the mean **`Age`** of
customers, with 10,000 resamples and `rng=1`.

```
95% CI: about 55.1  to  56.4
```

Then check it **by hand** with x̄ ± 2 · s/√n, as in §3.5.

```
by hand: about 55.08  to  56.39
```

In [ ]:
# Your code here

**5 · Write the sentence.** Turn your answer to question 4 into one sentence a manager could read.

_(your answer)_

**6 · Test it.** Somebody claims the average customer is **55** years old. Test that claim.

State H₀ and H₁ first, then run the test, then give your conclusion.

- **H₀:** _(your answer)_
- **H₁:** _(your answer)_

```
t ≈ 2.26,  p ≈ 0.0241
```

**Conclusion** — in the form from §4.4: _(your answer)_

In [ ]:
# Your code here

**7 · The honest follow-up.** Your test in question 6 rejects H₀. The sample mean age is 55.74
against a claim of 55.

Would you tell the brand this matters? **One sentence, and say why.**

_(your answer)_

---
# ⭐ Finished early?

**Optional and unassessed.**

In §4 we tested against 50,000 with a **two-sided** test. There are two other options —
`alternative='greater'` and `alternative='less'`.

Ask your AI:

> *"In scipy's ttest_1samp, what is the difference between a two-sided and a one-sided alternative,
> and when should I use each?"*

Then check what it tells you by running all three against `popmean=50000` and comparing the
p-values. In particular:

- What is the relationship between the two-sided p-value and the `'greater'` p-value?
- Which one would you have to decide on **before** looking at your data, and why does that matter?

_(your answer)_

In [ ]:
# Run all three here and compare

---
# ⭐ Optional · Sampling from a shape you choose

**Optional and unassessed.** From Jing Liu's notebook 5.

So far we sampled from a file. `numpy` can also draw random values from a **normal distribution** you
describe yourself. 📖 `rng.normal(loc=0, scale=1, size=500)` draws **500** values from a normal curve
with **mean 0** (`loc`) and **standard deviation 1** (`scale`).

In [ ]:
rng = np.random.RandomState(1)

x = rng.normal(loc=0, scale=1, size=500)

print(x.mean(), np.std(x, ddof=1))

**Your turn — four steps.**

1. Set the seed to **2025** and draw **500** values from a normal curve with mean **2** and sd **1**. Save
   them as `X1`. Print the mean and the standard deviation.
2. Is `X1` normal? Check with a histogram or a QQ plot (week 5).
3. 📖 `np.exp(X1)` raises e to the power of each value. Save it as `X2`. Print its mean and sd.
4. Is `X2` normal? Check it the same way.

```
X1: mean ≈ 1.94   sd ≈ 0.97
X2: mean ≈ 11.14  sd ≈ 13.37
```

_(your answer — what did `np.exp` do to the shape?)_

In [ ]:
# Your code here

---
# ✅ Done — hand it in

**No cell is designed to fail.** Run All should go straight through. Part 2's three-panel chart and
the bootstrap take a few seconds each — that is normal.

1. **Run All**, no red.
2. Save, **commit and sync**.
3. On Moodle: **this notebook file**.
4. **Stop your Codespace**.

## What you can now do

- [ ] explain why two samples from one population give different answers
- [ ] read a list comprehension that repeats something 1,000 times, and change the number in it
- [ ] say what the **central limit theorem** claims, and point at evidence for it
- [ ] compute a standard error two ways — from the formula, and by bootstrapping
- [ ] build a 95% confidence interval and **say what it means in one sentence**
- [ ] get a confidence interval by **trimming** the bootstrap means, and **by hand** with x̄ ± 2 SE
- [ ] say why a **smaller sample** gives a **wider** interval
- [ ] work out **t = (x̄ − μ) / SE** by hand, and say why |t| > 2 means p < 0.05 only from about 60 people
- [ ] say what a **p-value** is, in the textbook words and in plain words
- [ ] state H₀ and H₁, run a one-sample t-test, read t, p and the interval, and **write the conclusion sentence**
- [ ] explain why **p < 0.05 does not mean the difference matters**

## The midterm is next Wednesday

**14 Oct, 12:45 – 3:15 pm**, in our usual classroom.

- It is like our class exercises, but longer: one notebook in your Codespace, covering weeks 1–6.
  Some questions are multiple choice, some need a short written answer, and most are Python.
- Use your own laptop or a lab computer. **No mobile phones.**
- Open book, internet on, AI allowed. **No communication with anyone, online or offline.**
- Come from 12:30 with two Codespaces ready. Submit your .ipynb on Moodle by 3:15 pm.

To prepare:

- Read the slides for weeks 1–6.
- Redo the class exercises.
- Read the exercise recaps on Moodle.
- Pay attention to the non-coding questions too: the concepts, and what a result means.